# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.02473283  0.50751075 -0.78263065  0.09815859  0.102233  ]
 [ 0.40475325  0.77955205  0.1212864  -0.91129346  0.19444372]
 [ 0.99419362  0.14759915  0.45713084  0.99108701  0.11133062]
 [-0.02476026  0.52546496 -0.53805732 -0.191273   -0.70612836]
 [-0.59528387  0.96643909 -0.66917334 -0.00210478 -0.45880682]
 [-0.82501651 -0.48377565 -0.3695304   0.54225873 -0.82794632]
 [-0.03033812  0.53219985 -0.76061668 -0.65421336 -0.08215649]
 [ 0.12005198 -0.83744266 -0.43291859  0.89464088  0.24062601]
 [ 0.19745327 -0.9023426   0.43745001  0.47973743  0.14720159]
 [ 0.14166436 -0.7859547  -0.57887188 -0.23518317  0.22287756]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a1', 'a1', 'a2', 'a2', 'a1', 'a1', 'a1', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 0, 0, 0, 0, 1, 1, 0, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.07s/it]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.07s/it, loss=2745.6965]

SVI:   6%|▌         | 2/34 [00:01<00:34,  1.07s/it, loss=2986.3738]

SVI:   9%|▉         | 3/34 [00:01<00:33,  1.07s/it, loss=2387.4102]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.15it/s, loss=2531.3137]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.15it/s, loss=1498.1771]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.15it/s, loss=1810.2936]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.55it/s, loss=1810.2936]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.55it/s, loss=1185.8979]